# Etapa 1: Ingesta Raw y Validación Inicial

Este cuaderno realiza la validación inicial de presencia y consistencia de los archivos CSV originales de entrada colocados en la carpeta `data/raw/`.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # En Colab, el CWD suele ser /content, buscamos la carpeta del proyecto
    possible_roots = list(Path('/content').glob('*/notebooks'))
    if possible_roots:
        current_dir = possible_roots[0]
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
raw_dir = PROJECT_ROOT / "data" / "raw"
print("Ruta Raw de entrada:", raw_dir)
print("¿Se ejecuta en Google Colab?:", IN_COLAB)

## 1. Validación de Existencia de Archivos

In [ ]:
required_files = [
    "clientes.csv",
    "provincias.csv",
    "segmentos.csv",
    "productos.csv",
    "cliente_estado_mensual.csv",
    "cliente_producto_mensual.csv",
    "cliente_producto_alta.csv"
]

missing_files = []
for f in required_files:
    f_path = raw_dir / f
    if f_path.exists():
        print(f"✔ Encontrado: {f} (Tamaño: {f_path.stat().st_size / 1024:.2f} KB)")
    else:
        print(f"❌ FALTANTE: {f}")
        missing_files.append(f)

if len(missing_files) == 0:
    print("\n¡Todos los archivos requeridos están presentes en la capa Raw!")
else:
    print(f"\nERROR: Faltan {len(missing_files)} archivos en la capa Raw.")

## 2. Inspección Inicial de Dimensiones y Muestreo
Cargamos muestras de cada tabla para analizar el número de registros y verificar los nombres de las columnas.

In [ ]:
for f in required_files:
    f_path = raw_dir / f
    if f_path.exists():
        df = pd.read_csv(f_path, nrows=5)
        # Leer total de registros de forma rápida
        total_rows = sum(1 for _ in open(f_path)) - 1
        print("=" * 60)
        print(f"Tabla: {f} | Registros: {total_rows} | Columnas: {list(df.columns)}")
        display(df.head(2))